# Lab 5 network extension — Kundur CCT with ANDES

**Optional** follow-on after assessed Lab 5 on SMIB. Same SciML metrics (CCT, signed error, false-safe); ANDES is the network trajectory engine.

- Case: `kundur/kundur_full.xlsx`
- Official full bus-wise map (do not re-copy here): [Critical Clearing Time Mapping](https://docs.andes.app/en/stable/gallery/critical-clearing-time.html)
- Bridge page: `book/500_stability/03_andes_network_reference.md`
- Env: `conda activate sciml_ps_andes` (`andes==2.0.0`). From repo root: `pip install -e .`
- If ANDES warns about pycode: `andes prep`


In [ ]:
from pathlib import Path
import sys
import json
import numpy as np

ROOT = Path.cwd().resolve()
for c in [ROOT / "src", ROOT.parents[2] / "src" if len(ROOT.parents) >= 3 else None]:
    if c is not None and c.is_dir():
        sys.path.insert(0, str(c))
        break

from sciml_ps.andes_cct import (
    GALLERY_CCT_URL,
    KUNDUR_CASE,
    cct_sweep_one_bus,
    false_safe_count,
    signed_cct_error,
)

print("case:", KUNDUR_CASE)
print("gallery:", GALLERY_CCT_URL)


## Reference CCT on one bus (thin sweep)

Start with **one** fault bus and a short clearing-time grid. Expand toward the full gallery map only after this runs cleanly.


In [ ]:
# Short grid for teaching runtime; gallery uses a denser / wider sweep.
tc_values = np.arange(0.050, 0.351, 0.050)
ref = cct_sweep_one_bus(bus=8, t_fault=1.0, tc_values=tc_values, t_end=6.0)
t_ref = ref["cct_duration_s"]
print("andes", ref["andes_version"], "bus", ref["bus"], "CCT [s]", t_ref)
for row in ref["rows"]:
    print(f"  tc={row['clearing_duration']:.3f}s  {row['outcome']:14s}  max_sep={row['max_sep_deg']:.1f} deg")


## SciML-style metrics (no ML required for this cell)

Illustrate **signed CCT error** and **false-safe** counting with a crude predictor (optimistic fixed guess). Replace the predictor with your Lab 2 surrogate later; keep the same reference CCT and stability rule.


In [ ]:
# Optimistic guess: pretend CCT is 50 ms larger than truth (false-safe risk).
t_hat_optimistic = t_ref + 0.050
t_hat_conservative = max(t_ref - 0.050, 0.0)

err_opt = signed_cct_error(t_hat_optimistic, t_ref)
err_cons = signed_cct_error(t_hat_conservative, t_ref)
print("signed error optimistic [s]:", err_opt)
print("signed error conservative [s]:", err_cons)

t_ref_vec = np.array([t_ref, t_ref, t_ref])
t_hat_vec = np.array([t_hat_optimistic, t_hat_conservative, t_ref])
print("false-safe count (3 toy cases):", false_safe_count(t_hat_vec, t_ref_vec))


## Manifest snippet

Record version, case, fault bus, grid, and CCT for reproducibility (same habit as Lab 1).


In [ ]:
manifest = {
    "scenario_id": f"kundur_bus{ref['bus']}_cct",
    "case": ref["case"],
    "andes_version": ref["andes_version"],
    "bus": ref["bus"],
    "t_fault_s": ref["t_fault"],
    "tc_grid_s": ref["tc_values"],
    "cct_duration_s": t_ref,
    "stability_rule": "ANDES TDS criteria / angle separation (see gallery)",
    "gallery": ref["gallery"],
    "note": "Optional Lab 5 network extension; assessed Lab 5 remains SMIB.",
}
out = Path("lab05_network_cct_manifest.json")
out.write_text(json.dumps(manifest, indent=2), encoding="utf-8")
print("wrote", out.resolve())


## Next steps

1. Repeat for two more buses; build a small CCT table.
2. Replace the toy predictor with a learned surrogate (Lab 2) evaluated under the **same** clearing / label rule.
3. For the full heatmap / topology plots, follow the [ANDES CCT gallery](https://docs.andes.app/en/stable/gallery/critical-clearing-time.html) — summarize-then-link, do not paste that notebook into the course book.
